# **Optuna**


[Optuna](https://optuna.readthedocs.io/en/stable/index.html) is an open-source, automatic hyperparameter optimization (HPO) framework designed specifically for machine learning and deep learning. It is designed to automate the search for the best hyperparameters to maximize or minimize a model's target metric.

## 1. How Optuna Differs from Grid Search and Random Search

| Feature | Grid Search CV | Random Search CV | Optuna (Bayesian / TPE) |
| :-- | :-- | :-- | :-- |
| **Search Strategy** | **Exhaustive**: Evaluates every combination in a predefined grid. | **Random**: Evaluates random combinations from defined distributions. | **Informed**: Uses past trial results to choose the next best parameters. |
| **Efficiency** | Very slow; wastes time on obviously poor parameter combinations. | Faster than Grid Search, but still "blind" (doesn't learn as it goes). | Extremely fast; focuses resources on high-potential search spaces. |
| **Search Space** | Limited to discrete, predefined grids. | Can handle continuous ranges, but randomly. | Handles continuous, discrete, and nested conditional spaces dynamically. |
| **Early Stopping** | None (unless custom-built; evaluates all epochs/combinations). | None (evaluates all epochs/combinations). | **Pruning**: Stops poor trials early to save time. |

## 2. Core Components of Optuna

Optuna operates on a simple, imperative syntax defined by these main components:

### A. Study
A [Study](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.study.Study.html) represents an entire optimization session. You instantiate it using [create_study](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.create_study.html).
* *Example*: `study = optuna.create_study(direction="maximize")`

### B. Trial
A [Trial](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html) is a single evaluation of your model. Within a trial, Optuna "suggests" values for hyperparameters:
* `trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)`
* `trial.suggest_int('num_layers', 2, 4)`
* `trial.suggest_categorical('optimizer', ['adam', 'sgd'])`

### C. Objective Function
A user-defined Python function that represents the model training and evaluation process. It takes a `trial` object as input, suggests hyperparameters, trains the model, and returns a single score that Optuna uses to evaluate that trial.

### D. Sampler
The algorithm responsible for selecting the next set of hyperparameters based on history. Optuna's default sampler is the **TPESampler** (Tree-structured Parzen Estimator).

### E. Pruner
The algorithm responsible for automated early stopping (pruning). If a trial's intermediate results are significantly worse than previous successful trials, the pruner raises a `TrialPruned` exception and stops the trial immediately.

---

```python
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

def objective(trial):
    # selecting the parameters 
    n_estimators = trial.suggest_int('n_estimators', 50, 200, step=10)
    max_depth = trial.suggest_int('max_depth', 3, 20, step=4)

    # putting parameters in the Random Forest model
    model = RandomForestClassifier(
        n_estimators = n_estimators,
        max_depth = max_depth,
        random_state = 42
    )

    # creating cross val score
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

```

---

```python

# creating a study
    # using TPE sampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())

# optimizing the model
    # trials = 50
study.optimize(objective, n_trials=50)

```

---

```python
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')
```

---

```python
model = objective(study.best_trial)

history = model.fit(
    train_ds_fast,
    validation_data = val_ds_fast,
    epochs = 15
)

```